<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/earlier_version/kNN_modelling_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SECTION 1: Setup — load data, configure cross-validation
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic_knn_semiraw.csv'
df_knn = pd.read_csv(DATA_PATH)

X = df_knn.drop(columns=['attack_cat'])
y = df_knn['attack_cat']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("X shape:", X.shape, "| y shape:", y.shape)

Mounted at /content/drive
X shape: (150243, 34) | y shape: (150243,)


In [ ]:
# ============================================================
# SECTION 2: Build the leak-safe preprocessing pipeline —
# clamp, scale, bucket, encode — fit fresh on each fold's
# training data only. This is the full feature set, prior to
# any feature selection.
# ============================================================
clamp_cols = ['dload', 'dbytes', 'spkts', 'dmean', 'smean', 'sbytes',
              'dpkts', 'djit', 'sjit', 'sload', 'dur', 'synack',
              'sinpkt', 'dinpkt', 'response_body_len', 'ackdat']
nominal_cols = ['proto', 'state', 'service']

fold_data_full = []
for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for col in clamp_cols:
        lower, upper = X_train[col].quantile(0.01), X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower=lower, upper=upper)
        X_test[col] = X_test[col].clip(lower=lower, upper=upper)

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    proto_counts = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts[proto_counts >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data_full.append((X_train_enc, X_test_enc, y_train, y_test))
    print(f"Fold {fold_num+1}: prepared. X_train: {X_train_enc.shape}, kept protos: {keep_protos}")

Fold 1: prepared. X_train: (120194, 56), kept protos: ['tcp', 'udp']
Fold 2: prepared. X_train: (120194, 57), kept protos: ['tcp', 'udp']
Fold 3: prepared. X_train: (120194, 57), kept protos: ['tcp', 'udp']
Fold 4: prepared. X_train: (120195, 55), kept protos: ['tcp', 'udp']
Fold 5: prepared. X_train: (120195, 57), kept protos: ['tcp', 'udp']


In [ ]:
# ============================================================
# SECTION 3: Baseline and distance-weighted voting
# (k=5, Euclidean, full feature set)
# ============================================================
f1_baseline = [f1_score(y_test, KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean').fit(X_train, y_train).predict(X_test), average='macro')
               for X_train, X_test, y_train, y_test in fold_data_full]
print(f"Baseline (k=5, euclidean, uniform): {np.mean(f1_baseline):.4f} (± {np.std(f1_baseline):.4f})")

f1_weighted = [f1_score(y_test, KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean').fit(X_train, y_train).predict(X_test), average='macro')
               for X_train, X_test, y_train, y_test in fold_data_full]
print(f"Distance-weighted (k=5, euclidean): {np.mean(f1_weighted):.4f} (± {np.std(f1_weighted):.4f})")

Baseline (k=5, euclidean, uniform): 0.3836 (± 0.0084)
Distance-weighted (k=5, euclidean): 0.3909 (± 0.0050)


In [ ]:
# ============================================================
# SECTION 4: Full k / distance-metric grid search
# (single representative fold, distance-weighted voting fixed)
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full[0]
k_values = [1, 3, 5, 7, 9, 15, 21, 31]
metrics = ['euclidean', 'manhattan']

grid_results = []
for metric in metrics:
    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        model.fit(X_train_f, y_train_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results.append({'k': k, 'metric': metric, 'macro_f1': macro_f1})
        print(f"metric={metric:10s} k={k:3d}  macro-F1={macro_f1:.4f}")

best = pd.DataFrame(grid_results).sort_values('macro_f1', ascending=False).iloc[0]
print(f"\nBest (pre-resampling): metric={best['metric']}, k={best['k']}, macro-F1={best['macro_f1']:.4f}")

metric=euclidean  k=  1  macro-F1=0.3763
metric=euclidean  k=  3  macro-F1=0.3811
metric=euclidean  k=  5  macro-F1=0.3930
metric=euclidean  k=  7  macro-F1=0.3903
metric=euclidean  k=  9  macro-F1=0.3912
metric=euclidean  k= 15  macro-F1=0.3918
metric=euclidean  k= 21  macro-F1=0.3877
metric=euclidean  k= 31  macro-F1=0.3754
metric=manhattan  k=  1  macro-F1=0.3945
metric=manhattan  k=  3  macro-F1=0.4043
metric=manhattan  k=  5  macro-F1=0.4174
metric=manhattan  k=  7  macro-F1=0.4156
metric=manhattan  k=  9  macro-F1=0.4135
metric=manhattan  k= 15  macro-F1=0.4039
metric=manhattan  k= 21  macro-F1=0.3927
metric=manhattan  k= 31  macro-F1=0.3797

Best (pre-resampling): metric=manhattan, k=5, macro-F1=0.4174


Manhattan distance outperformed Euclidean at every k tested. k=5 was tentatively best on the non-resampled feature set; this is re-validated in Section 7 after the resampling strategy is finalised, since resampling can shift the optimal neighbourhood size.

In [ ]:
# ============================================================
# SECTION 5: Feature selection — screen then confirm
# ============================================================
mi_scores = mutual_info_classif(X_train_f, y_train_f, random_state=42)
mi_ranking = pd.Series(mi_scores, index=X_train_f.columns).sort_values(ascending=False)

for n in [10, 20, 30, 40]:
    top_n = mi_ranking.head(n).index.tolist()
    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_f[top_n], y_train_f)
    print(f"top {n:3d} features: macro-F1 = {f1_score(y_test_f, model.predict(X_test_f[top_n]), average='macro'):.4f}")

# Confirm top-10 with per-fold MI recomputation
fold_data_selected = []
fold_selected_features = []
for X_train, X_test, y_train, y_test in fold_data_full:
    mi = mutual_info_classif(X_train, y_train, random_state=42)
    ranking = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    top10 = ranking.head(10).index.tolist()
    fold_selected_features.append(top10)
    fold_data_selected.append((X_train[top10], X_test[top10], y_train, y_test))

f1_mi = [f1_score(y_test, KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan').fit(X_train, y_train).predict(X_test), average='macro')
         for X_train, X_test, y_train, y_test in fold_data_selected]
print(f"\nConfirmed top-10 (5-fold): {np.mean(f1_mi):.4f} (± {np.std(f1_mi):.4f})")
print("Selected features (identical across folds):", fold_selected_features[0])

top  10 features: macro-F1 = 0.4947
top  20 features: macro-F1 = 0.4275
top  30 features: macro-F1 = 0.4235
top  40 features: macro-F1 = 0.4140

Confirmed top-10 (5-fold): 0.4957 (± 0.0084)
Selected features (identical across folds): ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']


In [ ]:
# ============================================================
# SECTION 6: Resampling strategy comparison (k=7, Manhattan,
# ten-feature set)
# ============================================================
f1_flat, f1_smote, f1_ros, f1_tomek, f1_smote_tomek = [], [], [], [], []

for X_train, X_test, y_train, y_test in fold_data_selected:
    target_flat = {cls: max(count, 5000) for cls, count in y_train.value_counts().items()}
    Xs, ys = SMOTE(sampling_strategy=target_flat, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_flat.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan').fit(Xs, ys).predict(X_test), average='macro'))

    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    Xs, ys = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_smote.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan').fit(Xs, ys).predict(X_test), average='macro'))

    Xr, yr = RandomOverSampler(sampling_strategy=target_capped, random_state=42).fit_resample(X_train, y_train)
    f1_ros.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan').fit(Xr, yr).predict(X_test), average='macro'))

    Xt, yt = TomekLinks().fit_resample(X_train, y_train)
    f1_tomek.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan').fit(Xt, yt).predict(X_test), average='macro'))

    smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)
    f1_smote_tomek.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan').fit(Xst, yst).predict(X_test), average='macro'))

print(f"SMOTE, flat target:       {np.mean(f1_flat):.4f} (± {np.std(f1_flat):.4f})")
print(f"SMOTE, ratio-capped 5x:   {np.mean(f1_smote):.4f} (± {np.std(f1_smote):.4f})")
print(f"Random oversampling:      {np.mean(f1_ros):.4f} (± {np.std(f1_ros):.4f})")
print(f"Tomek links alone:        {np.mean(f1_tomek):.4f} (± {np.std(f1_tomek):.4f})")
print(f"SMOTE + Tomek:            {np.mean(f1_smote_tomek):.4f} (± {np.std(f1_smote_tomek):.4f})")

t_stat, p_val = stats.ttest_rel(f1_smote_tomek, f1_smote)
print(f"\nPaired t-test, SMOTE+Tomek vs SMOTE: t={t_stat:.4f}, p={p_val:.4f}")

SMOTE, flat target:       0.4919 (± 0.0074)
SMOTE, ratio-capped 5x:   0.5043 (± 0.0047)
Random oversampling:      0.5004 (± 0.0050)
Tomek links alone:        0.4917 (± 0.0081)
SMOTE + Tomek:            0.5060 (± 0.0118)

Paired t-test, SMOTE+Tomek vs SMOTE: t=0.3094, p=0.7725


In [ ]:
# ============================================================
# SECTION 7: Re-validation of k after adopting the resampling
# strategy (SMOTE+Tomek can shift the optimal neighbourhood size)
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_selected[0]
target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train_f, y_train_f)

for k in [5, 7]:
    model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='manhattan')
    model.fit(Xst, yst)
    print(f"k={k}: macro-F1 = {f1_score(y_test_f, model.predict(X_test_f), average='macro'):.4f}")

print("\nk=7 confirmed superior once resampling is applied; adopted as final.")

k=5: macro-F1 = 0.5095
k=7: macro-F1 = 0.5222

k=7 confirmed superior once resampling is applied; adopted as final.


In [ ]:
# ============================================================
# SECTION 8: Confirmation of feature scaling method
# (k=7, SMOTE+Tomek, ten-feature set)
# ============================================================
model_mm = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_mm.fit(Xst, yst)
print(f"Min-max: macro-F1 = {f1_score(y_test_f, model_mm.predict(X_test_f), average='macro'):.4f}")

scaler_z = StandardScaler()
Xst_z = scaler_z.fit_transform(Xst)
Xtest_z = scaler_z.transform(X_test_f)
model_z = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_z.fit(Xst_z, yst)
print(f"Z-score: macro-F1 = {f1_score(y_test_f, model_z.predict(Xtest_z), average='macro'):.4f}")
print("\nMin-max confirmed and retained.")

Min-max: macro-F1 = 0.5222
Z-score: macro-F1 = 0.5153

Min-max confirmed and retained.


In [ ]:
# ============================================================
# SECTION 9: FINAL MODEL — full 5-fold evaluation of the
# adopted configuration (Manhattan, k=7, ten MI-selected
# features, SMOTE+Tomek)
# ============================================================
final_f1, final_acc, final_weighted = [], [], []
all_y_test, all_y_pred = [], []

for X_train, X_test, y_train, y_test in fold_data_selected:
    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(Xst, yst)
    y_pred = model.predict(X_test)

    final_f1.append(f1_score(y_test, y_pred, average='macro'))
    final_acc.append(accuracy_score(y_test, y_pred))
    final_weighted.append(f1_score(y_test, y_pred, average='weighted'))
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

print(f"Mean macro-F1:    {np.mean(final_f1):.4f} (± {np.std(final_f1):.4f})")
print(f"Mean accuracy:    {np.mean(final_acc):.4f} (± {np.std(final_acc):.4f})")
print(f"Mean weighted-F1: {np.mean(final_weighted):.4f} (± {np.std(final_weighted):.4f})")
print()
print(classification_report(all_y_test, all_y_pred, digits=3))#

Mean macro-F1:    0.5060 (± 0.0118)
Mean accuracy:    0.7520 (± 0.0012)
Mean weighted-F1: 0.7634 (± 0.0041)

              precision    recall  f1-score   support

           0      0.926     0.827     0.874     83358
           1      0.696     0.729     0.712      8609
           2      0.201     0.368     0.260      1503
           3      0.290     0.427     0.346      5019
           4      0.767     0.758     0.762     26927
           5      0.139     0.091     0.110      1577
           6      0.497     0.648     0.563     19470
           7      0.399     0.562     0.467       169
           8      0.346     0.377     0.361      1426
           9      0.703     0.566     0.627      2185

    accuracy                          0.752    150243
   macro avg      0.496     0.535     0.508    150243
weighted avg      0.782     0.752     0.764    150243



In [ ]:
# ============================================================
# SECTION 10: Does clamping the 6 finally-selected clamped
# features actually help, at the FINAL configuration? Rebuilds
# the pipeline twice — once with clamping as usual, once
# skipping clamping only for the 6 features that survived into
# the final 10-feature set — everything else identical.
# ============================================================
final_10 = ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl',
            'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']
clamped_survivors = ['sbytes', 'dbytes', 'smean', 'dmean', 'dload', 'dinpkt', 'dpkts']

def build_final_pipeline(skip_clamp_cols):
    fold_data = []
    for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        active_clamp_cols = [c for c in clamp_cols if c not in skip_clamp_cols]
        for col in active_clamp_cols:
            lower, upper = X_train[col].quantile(0.01), X_train[col].quantile(0.99)
            X_train[col] = X_train[col].clip(lower=lower, upper=upper)
            X_test[col] = X_test[col].clip(lower=lower, upper=upper)

        numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
        scaler = MinMaxScaler()
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

        proto_counts = X_train['proto'].value_counts()
        threshold = 0.01 * len(X_train)
        keep_protos = proto_counts[proto_counts >= threshold].index.tolist()
        X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
        X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

        X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
        X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
        X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

        fold_data.append((X_train_enc[final_10], X_test_enc[final_10], y_train, y_test))
    return fold_data

def evaluate_final_config(fold_data):
    f1s = []
    for X_train, X_test, y_train, y_test in fold_data:
        target = {cls: min(count * 5, y_train.value_counts().max())
                  for cls, count in y_train.value_counts().items()}
        smote_st = SMOTE(sampling_strategy=target, random_state=42, k_neighbors=5)
        Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)
        m = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
        m.fit(Xst, yst)
        f1s.append(f1_score(y_test, m.predict(X_test), average='macro'))
    return f1s

fold_data_clamped = build_final_pipeline(skip_clamp_cols=[])
fold_data_unclamped = build_final_pipeline(skip_clamp_cols=clamped_survivors)

f1_clamped = evaluate_final_config(fold_data_clamped)
f1_unclamped = evaluate_final_config(fold_data_unclamped)

print(f"Clamped (final features):   {np.mean(f1_clamped):.4f} (± {np.std(f1_clamped):.4f})")
print(f"Unclamped (final features): {np.mean(f1_unclamped):.4f} (± {np.std(f1_unclamped):.4f})")

t_stat, p_val = stats.ttest_rel(f1_clamped, f1_unclamped)
print(f"Paired t-test: t={t_stat:.4f}, p={p_val:.4f}")

Clamped (final features):   0.5060 (± 0.0118)
Unclamped (final features): 0.5426 (± 0.0089)
Paired t-test: t=-5.0065, p=0.0075


In [ ]:
# ============================================================
# SECTION 11: SMOTE ratio sensitivity (2x, 3x, 5x, 10x) at
# the final configuration
# ============================================================
for ratio in [2, 3, 5, 10]:
    f1s = []
    for X_train, X_test, y_train, y_test in fold_data_selected:
        target = {cls: min(count * ratio, y_train.value_counts().max())
                   for cls, count in y_train.value_counts().items()}
        smote_st = SMOTE(sampling_strategy=target, random_state=42, k_neighbors=5)
        Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)
        m = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
        m.fit(Xst, yst)
        f1s.append(f1_score(y_test, m.predict(X_test), average='macro'))
    print(f"ratio={ratio}x: {np.mean(f1s):.4f} (± {np.std(f1s):.4f})")

ratio=2x: 0.5019 (± 0.0076)
ratio=3x: 0.4971 (± 0.0024)
ratio=5x: 0.5060 (± 0.0118)
ratio=10x: 0.4970 (± 0.0079)
